# 🧠 ***AD Speech Classification with Phi-4 Multimodal***

This notebook contains a training and evaluation pipeline for **Alzheimer’s Disease (AD) / dementia classification** from **speech audio + transcription** using the [`microsoft/Phi-4-multimodal-instruct`](https://huggingface.co/microsoft/Phi-4-multimodal-instruct) model.  

## 🔑 Key Features
- **Reproducibility**: Random seeds are set for Python, NumPy, and PyTorch.  
- **Custom Dataset Class** (`ADClassificationDataset`):  
  - Loads CSV files with `uid`, `transcription`, and `label` columns.  
  - Aligns each transcription with its corresponding audio file.  
  - Truncates overly long transcriptions and clips audio length to a maximum duration.  
  - Handles corrupted or missing audio safely by replacing with silent padding.  
- **Prompt-based classification**:  
  Each transcription is embedded into a natural language prompt asking the model to classify as `"dementia"` or `"control"`.  
- **Collation & Padding Functions**:  
  - `pad_sequence`, `cat_with_pad_original`, and `custom_collate_fn` ensure correct batching of text + audio embeddings.  
- **Model Loading**:  
  - Supports **Flash Attention 2** and low-memory loading.  
  - Compatible with LoRA adapters (if available).  
- **Training**:  
  - Uses Hugging Face’s `Trainer` with memory-efficient overrides.  
  - Supports gradient checkpointing, mixed precision (`fp16`/`bf16`), gradient accumulation, and distributed training with 🤗 Accelerate.  
- **Evaluation**:  
  - Custom `evaluate` function with stopping criteria for generation.  
  - Computes accuracy, precision, recall, F1 (weighted and dementia-specific), and BCE loss.  
  - Handles messy predictions by mapping variations back to `"dementia"` / `"control"`.  
- **Experiment Logging**:  
  - Logs metrics to TensorBoard.  
  - Saves best model checkpoints.  
  - Exports results (hyperparameters, metrics, predictions) to Excel.  

## 📂 Expected Inputs
- **Training CSV (`--train_csv_path`)**: must contain `uid`, `transcription`, `label`.  
- **Evaluation CSV (`--eval_csv_path`)**: same format.  
- **Audio Directories (`--train_audio_dir`, `--eval_audio_dir`)**: should contain `.wav`/`.flac` files named with the corresponding `uid`.  

## ⚙️ Main Arguments
- `--model_name_or_path` (default: `microsoft/Phi-4-multimodal-instruct`)  
- `--batch_size_per_gpu` (default: 1)  
- `--num_train_epochs` (default: 3)  
- `--learning_rate`, `--wd` (weight decay), `--gradient_accumulation_steps`  
- `--use_flash_attention` (flag)  
- `--mixed_precision` (`no` | `fp16` | `bf16`, default: `bf16`)  
- `--max_audio_seconds` (default: 30s per audio file)  

## 🚀 Workflow
1. **Pre-training evaluation** → run model on eval set before fine-tuning.  
2. **Fine-tuning** → train on AD speech dataset.  
3. **Final evaluation** → measure improvements.  
4. **Save outputs** → model, processor, and results stored in `--output_dir`.  

---

> 💡 **Tip**: Since this script was originally written for command-line use with `argparse`, you may want to wrap the `main()` call or replace `argparse` with notebook-friendly argument defaults when running inside Jupyter.


## ***Installing the requirements***

In [ ]:
!pip install -r requirements.txt

## ***Finetuning the Model***

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:32 accelerate launch \
    --mixed_precision bf16 \
    --num_processes 1 \
    --num_machines 1 \
    finetune.py \
    --model_name_or_path "microsoft/Phi-4-multimodal-instruct" \
    --train_csv_path "path/to/train.csv" \
    --train_audio_dir "path/to/train_audio/" \
    --eval_csv_path "path/to/test.csv" \
    --eval_audio_dir "path/to/test_audio/" \
    --output_dir "./phi4_ad_finetuned" \
    --num_train_epochs 3 \
    --batch_size_per_gpu 1 \
    --gradient_accumulation_steps 32 \
    --learning_rate 2e-5 \
    --logging_steps 10 \
    --save_steps 200 \
    --eval_steps 200 \
    --max_audio_seconds 70 \
    --use_flash_attention \
    --mixed_precision bf16 \
    --low_cpu_mem_usage \
    --seed 42


# 🧪 ***Testing the Fine-Tuned AD Speech Classifier***

The following cell provides a **testing script** for evaluating a **fine-tuned Phi-4 multimodal model** on a held-out test dataset. It is designed to be run **after training** to assess final model performance.  

## 🔑 Key Features
- **Device-aware Evaluation**: Ensures all tensors are correctly moved to CPU/GPU.  
- **Custom Evaluation Function** (`evaluate_with_device_handling`):  
  - Loads test dataset (`CSV + audio`).  
  - Uses the same `ADClassificationDataset` and prompt template as training.  
  - Generates predictions and extracts logits for `"dementia"` vs `"control"`.  
- **Metrics Computed**:  
  - Accuracy  
  - Weighted Precision, Recall, and F1  
  - Dementia-specific F1 (`f1_dementia`)  
  - Binary Cross-Entropy (BCE) Loss  
  - Number of evaluated samples  
- **Result Saving**:  
  - JSON with full metrics  
  - TXT summary of key results  
  - Excel file (`.xlsx`) with:
    - Test summary sheet  
    - Example predictions (up to 100 samples)  

## 📂 Expected Inputs
- **Fine-tuned model directory (`--model_dir`)**: Should contain saved model + processor from training.  
- **Test CSV (`--test_csv_path`)**: Contains `uid`, `transcription`, `label`.  
- **Test Audio Directory (`--test_audio_dir`)**: Folder with audio files corresponding to the UIDs.  

## ⚙️ Main Arguments
- `--model_dir` → directory of the fine-tuned model (required)  
- `--test_csv_path` → test CSV file (required)  
- `--test_audio_dir` → test audio directory (required)  
- `--output_dir` → where results are saved (default: `<model_dir>/test_results`)  
- `--max_test_samples` → optionally limit test set size  
- `--batch_size` (default: 1)  
- `--max_audio_seconds` (default: 30)  
- `--use_flash_attention` → flag to enable Flash Attention 2  
- `--mixed_precision` → `"no" | "fp16" | "bf16"` (default: `bf16`)  

## 🚀 Workflow
1. **Load fine-tuned model + processor**.  
2. **Create test dataset** from CSV + audio.  
3. **Run evaluation** with logits-based class probabilities.  
4. **Compute metrics** and save them in multiple formats.  
5. **Print test summary** with main metrics.  

---

> 💡 **Tip**: This script uses `argparse` for CLI execution. To run in a notebook, either:
> - Replace `argparse` with a dictionary of arguments, or  
> - Import `test_finetuned_model()` directly and call it programmatically with the correct paths.  


In [ ]:
!python test.py \
    --model_dir "./phi4_ad_finetuned" \
    --test_csv_path "path/to/test.csv" \
    --test_audio_dir "path/to/test_audio/" \
    --output_dir "./test_results" \
    --max_test_samples 100 \
    --batch_size 1 \
    --max_audio_seconds 70 \
    --use_flash_attention \
    --mixed_precision bf16